<a href="https://colab.research.google.com/github/saikala0109/AI-12-Days-Bootcamp/blob/main/dAY_6fn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic beautifulsoup4 requests

In [2]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from google import genai
from pydantic import BaseModel
from typing import List, Optional

In [4]:
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

In [5]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [6]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):

    try:
        r = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )

        r.raise_for_status()

        soup = BeautifulSoup(r.text, 'html.parser')

        # Remove script/style tags
        for tag in soup(['script', 'style']):
            tag.decompose()

        return soup.get_text(separator='\n', strip=True)[:max_chars]

    except Exception as e:
        print(f'Scrape failed: {e}')
        return None

In [8]:
test_url = "https://amazon.jobs/en/jobs/10399137/maintenance-technician"

text = fetch_jd(test_url)

if text:
    print(f"Got {len(text)} chars")
    print(text[:500])
else:
    print("Scraping failed")

Got 3744 chars
Maintenance Technician - Job ID: 10399137 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Accommodations
Benefits
Inclusive experiences
How We Hire
Leadership principles
Working at Amazon
FAQ
Maintenance Technician
Job ID: 10399137 | Amazon Commercial Services Pty Ltd
Apply now
Description
Amazon is seeking Maintenance Technician to join our Reliability, Maintenance & Engineering (RME) team


In [9]:
def normalise_jd(text: str) -> JD:

    resp = client.models.generate_content(

        model='gemini-2.5-flash',

        contents=f'''
Extract a JD JSON from this text.

{text}
''',

        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )

    return JD.model_validate_json(resp.text)

In [10]:
if text:

    jd = normalise_jd(text)

    print(jd.model_dump_json(indent=2))

{
  "company": "Amazon",
  "role": "Maintenance Technician",
  "must_have_skills": [
    "Formal Trade Qualifications: Certificate lll in Electrotechnology with current electrical trade license or Cert lll in Engineering OR Mechanical Trade (Fitter/Turner)",
    "Equivalent trade qualifications formally recognised in Australia",
    "Experience of planned preventative maintenance systems",
    "Experience fault finding within Material Handling Equipment/Automation systems",
    "Ability to read and understand electrical /mechanical drawings",
    "Experience of conveyor maintenance, motor controllers/inverters",
    "Experience of working to appropriate health & safety standards and regulations"
  ],
  "nice_to_have_skills": [
    "Experience with CMMS systems",
    "Experience with sortation machines",
    "Experience with maintaining/configuring bar code scanners",
    "Experience with print and apply machines"
  ],
  "min_cgpa": null,
  "locations": [
    "Jandakot, Perth, Australia

In [12]:
URLS = [

    "https://amazon.jobs/en/jobs/10399137/maintenance-technician",
    "https://www.linkedin.com/jobs/view/4417587848/?trackingId=TGhi8cS6RvWsTBaQHKGQLw%3D%3D",
    "https://www.linkedin.com/search/results/content/?keywords=JAVA%20DEVELOPER%20OBS&origin=GLOBAL_SEARCH_HEADER&postedBy=%5B%22following%22%5D",
    "URL_4",
    "URL_5"
]

jds = []

for url in URLS:

    text = fetch_jd(url)

    if text is None:
        continue

    try:

        jd = normalise_jd(text)

        jds.append(jd)

        print(f"✓ {jd.company} — {jd.role}")

    except Exception as e:

        print(f"✗ Failed: {e}")

print(f"\nProcessed {len(jds)} JDs")

✓ Amazon — Maintenance Technician
✓ SoftClouds — Java Full Stack Developer (Java+Vue3)
✓  — 
Scrape failed: Invalid URL 'URL_4': No scheme supplied. Perhaps you meant https://URL_4?
Scrape failed: Invalid URL 'URL_5': No scheme supplied. Perhaps you meant https://URL_5?

Processed 3 JDs
